# 07 · Fitting the models

**Paper:** Section IV-E (*Models*) and the importance measure of IV-F.

For every analysis, target and split, each model is fitted on the training set only and scores the
test set, for these feature sets:

* **B0** (the job request);
* **B0 + one group**, for each group;
* **B0 + one placebo**, for each of the 20 placebos.

| analysis | population | groups | baseline | models |
|---|---|---|---|---|
| primary | complete (R1) | the 10 groups | B0 | HGB, RF, LightGBM |
| secondary | GPU cohort | the 5 IPMI groups | B0 + `has_ganglia` | HGB |
| sensitivity | strict (R80) | the 10 groups | B0 | HGB, RF, LightGBM |

On the main split it also measures the **importance** of each group: the drop in test PR-AUC
when the group's four columns are permuted together in the B0 + group model (10 repeats, P7).

This notebook only fits and stores test scores (`data/predictions/`); notebook 08 evaluates
them. It takes about 20 minutes; existing results are reused (delete `data/predictions/` to refit).

In [1]:
import sys
sys.path.insert(0, "..")

import json
import time
import warnings

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score

from tcc.config import DATA, GROUPS, IPMI_GROUPS, N_PLACEBOS, SEED, group_cols

warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed")   # harmless, printed once per tree
PRED = DATA / "predictions"
PRED.mkdir(exist_ok=True)

## The models

Hyperparameters were fixed in advance, with no search (P5, P12). HGB decides; RF and LightGBM
verify. All three handle missing values natively.

In [2]:
def new_model(name):
    if name == "HGB":
        return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                              min_samples_leaf=50, l2_regularization=1.0,
                                              early_stopping=False, random_state=SEED)
    if name == "RF":
        return RandomForestClassifier(n_estimators=500, min_samples_leaf=20, max_features="sqrt",
                                      n_jobs=-1, random_state=SEED)
    if name == "LGBM":
        return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=50,
                              reg_lambda=1.0, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=SEED, deterministic=True, force_row_wise=True, verbose=-1)


def fit(name, X, y):
    # scikit-learn's worker threads can reset the warning filters, so they are set again before each fit
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
    m = new_model(name).fit(X, y)
    if name == "RF":
        # predict with one thread: with several, the order in which tree outputs are summed varies
        # and the score changes by ~1e-16, which can break ties differently from run to run
        m.set_params(n_jobs=1)
    return m

## Importance of a group (P7)

In [3]:
def permutation_importance(model, X, y, cols, seed=SEED, repeats=10):
    """Mean drop in PR-AUC when the columns `cols` are permuted together."""
    base = average_precision_score(y, model.predict_proba(X)[:, 1])
    rng = np.random.default_rng(seed)
    drops = []
    for _ in range(repeats):
        Xp = X.copy()
        Xp[:, cols] = X[rng.permutation(len(X))][:, cols]
        drops.append(base - average_precision_score(y, model.predict_proba(Xp)[:, 1]))
    return float(np.mean(drops))

## The analyses

In [4]:
F = pd.read_parquet(DATA / "features.parquet").set_index("job_id")
B0 = pd.read_csv(DATA / "b0_columns.csv").column.tolist()
ANALYSES = {
    "primary":     {"population": "complete", "groups": list(GROUPS), "baseline": B0,
                    "models": ["HGB", "RF", "LGBM"], "placebo": "placebo_complete"},
    "secondary":   {"population": "cohort", "groups": IPMI_GROUPS, "baseline": B0 + ["has_ganglia"],
                    "models": ["HGB"], "placebo": "placebo_cohort"},
    "sensitivity": {"population": "strict", "groups": list(GROUPS), "baseline": B0,
                    "models": ["HGB", "RF", "LGBM"], "placebo": "placebo_strict"},
}


def feature_sets(a):
    sets = {"B0": []}
    sets.update({g: group_cols(g) for g in a["groups"]})
    sets.update({f"placebo{k:02d}": group_cols(f"{a['placebo']}{k:02d}") for k in range(N_PLACEBOS)})
    return sets

## Fit everything

In [5]:
for name, a in ANALYSES.items():
    splits = json.loads((DATA / f"splits_{a['population']}.json").read_text())
    for target in ("a", "b"):
        out_pred, out_imp = PRED / f"{name}_{target}.parquet", PRED / f"importance_{name}_{target}.parquet"
        if out_pred.exists() and out_imp.exists():
            print(f"{name}/{target}: already fitted")
            continue
        y_col, preds, imps, start = f"target_{target}", [], [], time.time()
        for split in ("main", "block1", "block2", "block3"):
            train, test = F.loc[splits[split]["train"]], F.loc[splits[split]["test"]]
            train, test = train[train[y_col].notna()], test[test[y_col].notna()]   # only the target's states
            y_train, y_test = train[y_col].values.astype(int), test[y_col].values.astype(int)
            for model in a["models"]:
                for fs, cols in feature_sets(a).items():
                    X_cols = a["baseline"] + cols
                    X_train, X_test = train[X_cols].to_numpy(float), test[X_cols].to_numpy(float)
                    m = fit(model, X_train, y_train)
                    preds.append(pd.DataFrame({"job_id": test.index.values, "split": split, "model": model,
                                               "feature_set": fs, "score": m.predict_proba(X_test)[:, 1].astype(np.float32)}))
                    if split == "main" and fs != "B0":
                        idx = list(range(len(a["baseline"]), len(X_cols)))
                        imps.append({"model": model, "feature_set": fs,
                                     "importance": permutation_importance(m, X_test, y_test, idx)})
            print(f"{name}/{target}/{split}: train {len(train):,} ({y_train.sum()} pos), "
                  f"test {len(test):,} ({y_test.sum()} pos)   {time.time() - start:.0f} s")
        pd.concat(preds, ignore_index=True).to_parquet(out_pred, index=False)
        pd.DataFrame(imps).to_parquet(out_imp, index=False)

primary/a/main: train 16,593 (2556 pos), test 4,422 (608 pos)   144 s


primary/a/block1: train 5,233 (480 pos), test 1,405 (575 pos)   195 s


primary/a/block2: train 6,637 (828 pos), test 1,318 (227 pos)   248 s


primary/a/block3: train 5,118 (930 pos), test 1,304 (124 pos)   299 s


primary/b/main: train 16,730 (2693 pos), test 4,213 (399 pos)   142 s


primary/b/block1: train 5,526 (773 pos), test 927 (97 pos)   193 s


... (14 outputs left out)

sensitivity/b/main: train 7,133 (633 pos), test 1,853 (104 pos)   87 s


sensitivity/b/block1: train 2,518 (195 pos), test 432 (43 pos)   134 s


sensitivity/b/block2: train 2,486 (217 pos), test 648 (53 pos)   180 s


sensitivity/b/block3: train 2,290 (181 pos), test 612 (48 pos)   225 s
